# Notebook 1: STAC — Discovery

Search a STAC catalog by bbox and time, inspect items and assets, and sign URLs for Planetary Computer.

**Dependencies:** `pystac-client`, `planetary-computer`

In [ ]:
import pystac_client
import planetary_computer

## Open the catalog

Microsoft Planetary Computer STAC API (no API key required for read).

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)

## List collections (optional)

In [ ]:
collections = list(catalog.get_collections())
print(f"Total collections: {len(collections)}")
for col in collections[:15]:
    print(col.id)

## Search by bbox and datetime

Use Sentinel-2 L2A over a small area (e.g. San Francisco Bay) and one month.

In [ ]:
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[-122.5, 37.2, -122.0, 37.6],  # [min_lon, min_lat, max_lon, max_lat]
    datetime="2023-06-01/2023-06-30",
    max_items=10,
)
items = list(search.items())
print(f"Found {len(items)} items")

## Inspect one item

Each item has `id`, `datetime`, `bbox`, `geometry`, and `assets` (dict of asset key → Asset with `href`).

In [ ]:
item = items[0]
print("Item ID:", item.id)
print("Datetime:", item.datetime)
print("Bbox:", item.bbox)
print("\nAssets:")
for key, asset in item.assets.items():
    print(f"  {key}: {asset.href[:80]}...")

## Sign items (Planetary Computer)

Signed URLs are required for access. Sign the item so asset `href`s can be used by rioxarray/rasterio.

In [ ]:
signed_item = planetary_computer.sign(item)
# Example: get signed href for red band (B04)
b04_href = signed_item.assets["B04"].href
print("Signed B04 href (first 100 chars):", b04_href[:100])

## Sign all items for later use

In the end-to-end workflow we sign all items, then open assets by href.

In [ ]:
signed_items = [planetary_computer.sign(it) for it in items]
print(f"Signed {len(signed_items)} items. Ready for rioxarray.open_rasterio(asset.href).")